# Intrinsics Check: Z4 vs CCD Height Map (v2)

**Author:** Aaron Roodman  
**Date Created:** 2026-04-20  
**Last Modified:** 2026-04-20  
**Status:** In Progress  
**Keywords:** AOS, Intrinsic Wavefront, Z4, Defocus, CCD Height Map, Full Array Mode  

## Description

Check whether the Z4 intrinsic wavefront map (as produced by the AOS pipeline and
stored in the FAM mktable HDF5 output) correctly accounts for the focal plane
CCD-to-CCD height variation.

Key functionality:
1. Read the FAM donuts + visits HDF5 produced by `intrinsics_mktable` (or the `mktable` step of `run-pipeline`)
2. Read the matching `_fits.parquet` file with the per-visit linear (k1, k2, k3) fit coefficients
3. Subtract the linear (tilt/tip + piston) portion of Z4 per donut using OCS field angles
4. Bin the linear-corrected Z4 in focal plane (fpx, fpy) and plot the median map
5. Build a custom Z4 intrinsic per donut using **`ts_wep`'s `Instrument.getIntrinsicZernikes`** (evaluated on a field-angle grid and interpolated for speed), and add a CCD height-map correction (15 μm Z4 per mm of local piston). The `ts_wep` develop branch does **not** apply a per-CCD height map inside `getIntrinsicZernikes` — its `batoidModel` only carries a uniform `defocalOffset` — so we add the height term ourselves here.
6. Plot the custom Z4 intrinsic, the difference vs the linear-corrected data, and direct comparisons against the pipeline `zk_intrinsic` already stored in the HDF5.

**Output:** In-notebook plots only (median maps, difference map, hexbin and 2D comparisons).

**Based on:**
- `~/Astrophysics/Code/Claude/intrinsic_wavefront.ipynb` — invoking `Instrument.getIntrinsicZernikes` from `lsst.ts.wep`
- `~/Astrophysics/Code/Claude/plot_Z_FAM_August-archive.ipynb` — focal-plane height map reader, Z4↔height conversion factor
- `intrinsics_mktable.ipynb` / `code/intrinsics_lib.py` — HDF5 donut table format, `get_intrinsic_map` pattern
- `code/dz_fitting.py` — focal plane Zernike basis used by the `_fits.parquet` fits (fp_radius = 1.75°, basis normalized to unit disk)

<a id='changelog'></a>
## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-04-20 | Aaron Roodman | Initial version — radial i-band polynomial Z4 intrinsic + KNN height map |
| 2026-04-20 | Aaron Roodman | v2: replaced radial polynomial with `ts_wep` `Instrument.getIntrinsicZernikes` evaluated on a field-angle grid and interpolated; wrapped the custom intrinsic in `build_custom_z4_intrinsic`. Added per-visit HDF5 loader workaround for the pandas shape-mismatch in `read_donuts_table`. |

## Table of Contents

1. [Parameters](#params)
2. [Setup & Imports](#setup)
3. [Helper Functions](#functions)
4. [Data Access](#data)
5. [Analysis](#analysis)
6. [Results & Plots](#results)

<a id='params'></a>
## Parameters

In [ ]:
# ============================================================
# Parameters — All configurable values collected here
# ============================================================

# Input HDF5 from intrinsics_mktable (streaming format)
INPUT_HDF5 = 'output/aos_fam_danish_triplets_20260315_20260317.hdf5'

# Corresponding linear-fit parquet (auto-derived as {stem}_fits.parquet if None)
FIT_PARQUET = None

# CCD focal plane height map (on USDF, copied under notebooks/rubin-work/aos/output/)
HEIGHT_MAP_FITS = 'output/LSST_FP_cold_b_measurement_4col_bysurface.fits'

# Coordinate system used for the linear fit (must match the fit parquet)
COORD_SYS = 'OCS'

# Focal-plane radius used for normalizing the Zernike basis in the fit
# (matches default in code/dz_fitting.py: focal_plane_zernike_basis)
FP_RADIUS_DEG = 1.75

# ts_wep intrinsic grid — field angle range (deg) and grid steps per axis.
# getIntrinsicZernikes is expensive; we evaluate once on this grid and
# interpolate to each donut's (thx, thy) with scipy.LinearNDInterpolator.
INTRINSIC_GRID_EXTENT_DEG = 1.85   # slightly > FP radius so edges are inside
INTRINSIC_GRID_N = 81              # ≈0.046° ≈ 165 arcsec spacing

# Photometric band for the intrinsic (BandLabel.LSST_I by default; see below).
INTRINSIC_BAND_NAME = 'LSST_I'

# Height-to-Z4 conversion: μm of Z4 per mm of local piston (height).
# Guillem's estimate (≈0.15 μm Z4 per 10 μm defocus = 15 μm/mm) — same as old notebook.
# NOTE: ts_wep/batoid's getIntrinsicZernikes does NOT apply a CCD-by-CCD height
# map — we add this contribution ourselves.
HEIGHT_TO_Z4_UM_PER_MM = 15.0

# Binning for the focal-plane (fpx, fpy) median maps
FP_NSTEPS = 257           # number of bin edges (fine pitch ≈ 2.5 mm/bin)
FP_MAX_MM = 320.0         # half-range in mm

# Whether to restrict to donuts whose intra and extra centroids matched
REQUIRE_MATCHED = True

# Output PDF: all plots from the Results section go here
PDF_OUTPUT = 'output/intrinsics_checkZ4.pdf'

<a id='setup'></a>
## Setup & Imports

In [ ]:
import os
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from mpl_toolkits import axes_grid1

from astropy.io import fits
from astropy.table import QTable, Table
from scipy.stats import binned_statistic_2d
from scipy.interpolate import LinearNDInterpolator
from sklearn.neighbors import KNeighborsRegressor
import tqdm

# LSST camera (for pixel → focal-plane transform)
from lsst.obs.lsst import LsstCam

# ts_wep — source of the intrinsic Zernike model
from lsst.ts.wep.task import (
    CalcZernikesTask, CalcZernikesTaskConfig, EstimateZernikesDanishTask,
)
from lsst.ts.wep.utils import getTaskInstrument, BandLabel

# Shared cameraGeom helpers (common/camera_utils.py)
sys.path.insert(0, str(Path.cwd().parent))
from common.camera_utils import pixel_to_focal  # noqa: E402

plt.rcParams['figure.dpi'] = 110

<a id='functions'></a>
## Helper Functions

In [ ]:
def add_colorbar(im, aspect=20, pad_fraction=0.5, **kwargs):
    """Attach a vertical colorbar that matches the host axes' height."""
    divider = axes_grid1.make_axes_locatable(im.axes)
    width = axes_grid1.axes_size.AxesY(im.axes, aspect=1.0 / aspect)
    pad = axes_grid1.axes_size.Fraction(pad_fraction, width)
    current_ax = plt.gca()
    cax = divider.append_axes('right', size=width, pad=pad)
    plt.sca(current_ax)
    return im.axes.figure.colorbar(im, cax=cax, **kwargs)


# ------------------------------------------------------------------
# Donut → focal-plane coordinates
# ------------------------------------------------------------------

def compute_fp_coords(aosTable, camera, x_col='centroid_x_intra',
                      y_col='centroid_y_intra'):
    """Evaluate fpx/fpy in mm for every donut using per-detector transforms.

    Groups donuts by detector and calls ``common.camera_utils.pixel_to_focal``
    once per sensor (vectorized over all donuts on that sensor) rather than
    per-row, which is much faster.

    Parameters
    ----------
    aosTable : astropy.table.QTable
        Donut table with `detector` (string) and pixel centroid columns.
    camera : lsst.afw.cameraGeom.Camera
        Camera geometry (e.g. `LsstCam.getCamera()`).
    x_col, y_col : str
        Pixel centroid column names (default: intra-focal centroids).

    Returns
    -------
    fpx_mm, fpy_mm : ndarray
        Focal-plane position per donut, in mm.
    """
    det_names = np.asarray(aosTable['detector']).astype(str)
    x_pix = np.asarray(aosTable[x_col], dtype=float)
    y_pix = np.asarray(aosTable[y_col], dtype=float)

    fpx = np.full_like(x_pix, np.nan)
    fpy = np.full_like(y_pix, np.nan)

    for det in camera:
        name = det.getName()
        mask = (det_names == name)
        if not np.any(mask):
            continue
        fx, fy = pixel_to_focal(x_pix[mask], y_pix[mask], det)
        fpx[mask] = fx
        fpy[mask] = fy
    return fpx, fpy


def ccd_center_fp(camera):
    """Return {detector_name: (cx_mm, cy_mm)} for every detector in `camera`.

    The center is the focal-plane position of each CCD's bounding-box center
    pixel (per lsst.afw.cameraGeom), converted to mm via pixel_to_focal —
    matching the snippet
    ``camera[idet].getBBox().getCenter()`` + ``pixel_to_focal(...)``.
    """
    centers = {}
    for det in camera:
        c = det.getBBox().getCenter()  # lsst.geom.Point2D
        cx, cy = pixel_to_focal(np.array([c.getX()]),
                                np.array([c.getY()]), det)
        centers[det.getName()] = (float(cx[0]), float(cy[0]))
    return centers


def build_transposed_pipeline_z4(fpx_mm, fpy_mm, det_names,
                                 Z4_pipeline_intrinsic_um,
                                 camera, keep_mask=None):
    """Transpose x↔y around each CCD's center, then look up the pipeline Z4
    intrinsic at the swapped position.

    Diagnostic for a suspected (fpx, fpy) axis swap in the pipeline's
    intrinsic-Zernike interpolation: if the pipeline intrinsic only matches a
    reference (custom) intrinsic after this transposition, the axes are indeed
    swapped.

    For each donut at ``(fpx, fpy)`` on CCD ``d`` with focal-plane center
    ``(cx, cy)``, the swapped position is
    ``(cx + (fpy - cy), cy + (fpx - cx))`` — i.e. the within-CCD offset has
    its x and y components exchanged. The pipeline intrinsic at that location
    is read by nearest-neighbor lookup over the real donut positions (cKDTree)
    using only the ``keep_mask`` subset as the reference set.

    Parameters
    ----------
    fpx_mm, fpy_mm : ndarray
        Per-donut focal-plane position in mm.
    det_names : ndarray of str
        Per-donut detector name (e.g. 'R22_S11').
    Z4_pipeline_intrinsic_um : ndarray
        Per-donut pipeline Z4 intrinsic in μm.
    camera : lsst.afw.cameraGeom.Camera
    keep_mask : ndarray of bool or None
        Restrict the KDTree reference set to these donuts (e.g. ``keep``).
        If None, use all donuts.

    Returns
    -------
    Z4_transposed_um : ndarray
        Transposed pipeline Z4 intrinsic per donut, μm.
    fpx_swap, fpy_swap : ndarray
        Per-donut swapped focal-plane position (mm).
    det_centers : dict
        {detector_name: (cx_mm, cy_mm)} for diagnostic use.
    """
    from scipy.spatial import cKDTree

    det_centers = ccd_center_fp(camera)

    fpx_swap = np.full_like(fpx_mm, np.nan)
    fpy_swap = np.full_like(fpy_mm, np.nan)
    for name, (cx, cy) in det_centers.items():
        mask = (det_names == name)
        if not np.any(mask):
            continue
        dx = fpx_mm[mask] - cx
        dy = fpy_mm[mask] - cy
        fpx_swap[mask] = cx + dy
        fpy_swap[mask] = cy + dx

    if keep_mask is None:
        keep_mask = np.ones(len(fpx_mm), dtype=bool)
    ref_pts = np.column_stack([fpx_mm[keep_mask], fpy_mm[keep_mask]])
    ref_vals = Z4_pipeline_intrinsic_um[keep_mask]
    tree = cKDTree(ref_pts)
    _, nn_idx = tree.query(np.column_stack([fpx_swap, fpy_swap]))
    Z4_transposed = ref_vals[nn_idx]
    return Z4_transposed, fpx_swap, fpy_swap, det_centers


# ------------------------------------------------------------------
# Focal-plane height map
# ------------------------------------------------------------------

def make_metrology_table(file, rsid=None):
    """Build a per-point focal-plane metrology table from the height map FITS file.

    Mirrors the reader in plot_Z_FAM_August-archive.ipynb: each per-sensor BinTableHDU
    is concatenated into one table with columns (fpx, fpy, z_mod, z_meas, det).
    Note the x/y swap from CCS to focal-plane convention: fpx = Y_CCS, fpy = X_CCS.
    """
    rows = []
    with fits.open(file) as hdulist:
        for hdu in tqdm.tqdm(hdulist, desc='metrology HDUs'):
            if not isinstance(hdu, fits.BinTableHDU):
                continue
            extname = hdu.header.get('EXTNAME', '')
            if rsid is not None and extname != rsid:
                continue
            if rsid is None and not re.fullmatch(r'R\d\dS\d\d', extname):
                continue
            det_label = re.sub(r'(R\d\d)(S\d\d)', r'\1_\2', extname)
            tab = Table(hdu.data)
            for x, y, z_mod, z_meas in zip(
                tab['X_CCS'], tab['Y_CCS'], tab['Z_CCS_MODEL'], tab['Z_CCS_MEASURED']
            ):
                fpx = y  # fpx = Y_CCS
                fpy = x  # fpy = X_CCS
                rows.append([fpx, fpy, z_mod, z_meas, det_label])
    return Table(rows=rows, names=['fpx', 'fpy', 'z_mod', 'z_meas', 'det'])


def get_height_interpolator(metrology_table, k=3, weight_type='distance', ztype='measured'):
    """KNN interpolator for focal-plane height given a metrology table."""
    x = np.column_stack((metrology_table['fpx'], metrology_table['fpy']))
    if ztype == 'measured':
        y = np.array(metrology_table['z_meas'])
    elif ztype == 'model':
        y = np.array(metrology_table['z_mod'])
    else:
        raise ValueError("ztype must be 'measured' or 'model'")
    knn = KNeighborsRegressor(n_neighbors=k, weights=weight_type)
    knn.fit(x, y)

    def interp_func(fpx, fpy):
        fpx = np.atleast_1d(fpx)
        fpy = np.atleast_1d(fpy)
        pts = np.column_stack((fpx, fpy))
        return knn.predict(pts)

    return interp_func


# ------------------------------------------------------------------
# Linear (k1, k2, k3) model evaluation
# ------------------------------------------------------------------

def linear_fit_values(thx_deg, thy_deg, k1, k2, k3, fp_radius=FP_RADIUS_DEG):
    """Evaluate the Z1-Z3 focal-plane Zernike linear model at each donut.

    Matches the basis in code/dz_fitting.focal_plane_zernike_basis:
        Z1 = 1            (piston)
        Z2 = 2 * x        (tilt), x = thx_deg / fp_radius
        Z3 = 2 * y        (tip),  y = thy_deg / fp_radius
    So the linear model value is:
        k1 * 1 + k2 * 2*x + k3 * 2*y  (μm)
    """
    x = thx_deg / fp_radius
    y = thy_deg / fp_radius
    return k1 + 2.0 * k2 * x + 2.0 * k3 * y


# ------------------------------------------------------------------
# ts_wep intrinsic Zernike interpolator
# ------------------------------------------------------------------

def _make_ts_wep_instrument(camera):
    """Return a ts_wep Instrument configured as in intrinsic_wavefront.ipynb
    and intrinsics_lib.get_intrinsic_map.
    """
    config = CalcZernikesTaskConfig()
    config.estimateZernikes.retarget(EstimateZernikesDanishTask)
    config.donutStampSelector.maxSelect = 20
    config.donutStampSelector.maxFracBadPixels = 2.0e-4
    config.donutStampSelector.useCustomSnLimit = True
    config.donutStampSelector.minSignalToNoise = 100
    config.estimateZernikes.binning = 2
    config.estimateZernikes.saveHistory = False

    task = CalcZernikesTask(config=config)

    # Pick any real science detector name — the instrument object only uses
    # it to look up the pupil geometry, not its focal-plane position.
    camera_id_map = camera.getIdMap()
    detector_name = camera_id_map[195].getName()
    instrument = getTaskInstrument(
        'LSSTCam', detector_name,
        task.estimateZernikes.config.instConfigFile,
    )
    return instrument


def build_z4_intrinsic_interpolator(iZs, band=BandLabel.LSST_I,
                                    extent_deg=INTRINSIC_GRID_EXTENT_DEG,
                                    n_grid=INTRINSIC_GRID_N, verbose=True):
    """Evaluate ts_wep `Instrument.getIntrinsicZernikes` on a 2-D field-angle
    grid and return a `scipy.interpolate.LinearNDInterpolator` for the Z4 term
    (converted to μm).

    Rationale: a single `getIntrinsicZernikes` call is slow (batoid ray-trace),
    so calling it per donut (~5×10⁵ calls) is impractical. Evaluating on a
    fine (thx, thy) grid and interpolating gives sub-μm accuracy for Z4 in
    under a minute.

    Parameters
    ----------
    iZs : sequence of int
        Noll indices matching the data (e.g. [4, 5, ..., 26]).
    band : BandLabel
        Filter band for the intrinsic (default `BandLabel.LSST_I`).
    extent_deg : float
        Half-range of the square grid in degrees (should cover the FP).
    n_grid : int
        Number of grid points per axis.
    verbose : bool
        Whether to print progress and grid info.

    Returns
    -------
    z4_um_interp : callable
        Takes (thx_deg, thy_deg) and returns Z4 intrinsic in μm.
    debug : dict
        {'X', 'Y', 'zkIntrinsics_m'} — the raw grid (for diagnostic plots).
    """
    if 4 not in iZs:
        raise ValueError('iZs must contain Z4 (Noll index 4)')
    z4_col = list(iZs).index(4)

    camera = LsstCam.getCamera()
    instrument = _make_ts_wep_instrument(camera)

    xs = np.linspace(-extent_deg, extent_deg, n_grid)
    ys = np.linspace(-extent_deg, extent_deg, n_grid)
    XX, YY = np.meshgrid(xs, ys)
    RR = np.sqrt(XX**2 + YY**2)
    mask = RR <= (extent_deg + 1e-6)
    X = XX[mask].flatten()
    Y = YY[mask].flatten()

    nZk = len(iZs)
    nPts = len(X)
    zkIntrinsics_m = np.zeros((nZk, nPts))

    if verbose:
        print(f'Evaluating ts_wep getIntrinsicZernikes on {nPts} field-angle '
              f'grid points (band={band.name}, {nZk} Noll terms)…')

    it = range(nPts)
    if verbose:
        it = tqdm.tqdm(it, desc='ts_wep intrinsic')
    noll_arr = np.asarray(iZs, dtype=int)
    for i in it:
        zkIntrinsics_m[:, i] = instrument.getIntrinsicZernikes(
            xAngle=float(X[i]), yAngle=float(Y[i]),
            defocalType=None, band=band, nollIndices=noll_arr,
        )

    # Build the Z4 interpolator (convert meters → μm)
    pts = np.column_stack([X, Y])
    z4_m = zkIntrinsics_m[z4_col, :]
    interp = LinearNDInterpolator(pts, z4_m * 1.0e6, fill_value=np.nan)

    def z4_um_interp(thx_deg, thy_deg):
        return interp(np.atleast_1d(thx_deg), np.atleast_1d(thy_deg))

    return z4_um_interp, {'X': X, 'Y': Y, 'zkIntrinsics_m': zkIntrinsics_m}


def build_custom_z4_intrinsic(thx_deg, thy_deg, fpx_mm, fpy_mm,
                              z4_um_interp, height_interp,
                              height_to_z4_um_per_mm=HEIGHT_TO_Z4_UM_PER_MM):
    """Return a per-donut custom Z4 intrinsic (μm).

    The custom intrinsic is the sum of two parts:

    1. ``z4_um_interp(thx_deg, thy_deg)`` — ts_wep's
       ``Instrument.getIntrinsicZernikes`` evaluated on a grid and
       linearly interpolated (μm).
    2. ``height_to_z4_um_per_mm * height_interp(fpx_mm, fpy_mm)`` —
       a piston-per-height contribution built from the CCD focal-plane
       metrology map. This is added here because ts_wep/batoid's
       `getIntrinsicZernikes` does **not** incorporate the CCD height map.

    Parameters
    ----------
    thx_deg, thy_deg : ndarray
        Field angles in degrees per donut (OCS or CCS both fine — the
        intrinsic is field-position dependent, not rotator dependent).
    fpx_mm, fpy_mm : ndarray
        Donut focal-plane position in mm (any consistent CCS/DVCS choice
        that matches the metrology map).
    z4_um_interp : callable
        As returned by ``build_z4_intrinsic_interpolator``.
    height_interp : callable
        As returned by ``get_height_interpolator`` (returns height in mm).
    height_to_z4_um_per_mm : float
        Conversion factor, μm of Z4 per mm of local piston.

    Returns
    -------
    Z4_um : ndarray
        Custom Z4 intrinsic per donut (μm).
    parts : dict
        {'optics': Z4 from optics model (μm), 'height': Z4 from height
        map (μm)} — for diagnostic plotting.
    """
    z4_optics_um = z4_um_interp(thx_deg, thy_deg)
    z4_height_um = height_to_z4_um_per_mm * height_interp(fpx_mm, fpy_mm)
    return z4_optics_um + z4_height_um, {
        'optics': z4_optics_um,
        'height': z4_height_um,
    }


# ------------------------------------------------------------------
# Binning + plotting helpers
# ------------------------------------------------------------------

def bin_median_2d(xval, yval, zval, xbins, ybins):
    """Median of zval on an (xbins × ybins) 2D grid over (xval, yval)."""
    stat, x_edges, y_edges, _ = binned_statistic_2d(
        xval, yval, zval, statistic='median', bins=[xbins, ybins]
    )
    return stat, x_edges, y_edges


def plot_fp_map(stat, x_edges, y_edges, title, cmap='viridis',
                vmin=None, vmax=None, label=None):
    """Imshow a (fpx, fpy) 2D map with a matched colorbar."""
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(
        stat.T, origin='lower',
        extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
        cmap=cmap, interpolation='none', aspect='equal',
        vmin=vmin, vmax=vmax,
    )
    ax.set_xlabel('fpx [mm]')
    ax.set_ylabel('fpy [mm]')
    ax.set_title(title)
    cb = add_colorbar(im)
    if label is not None:
        cb.set_label(label)
    fig.tight_layout()
    return fig, ax

<a id='data'></a>
## Data Access

In [ ]:
# Resolve paths and load visits + donuts from HDF5 (astropy HDF5 format)
hdf5_path = Path(INPUT_HDF5)
if not hdf5_path.exists():
    raise FileNotFoundError(f'HDF5 not found: {hdf5_path}')

if FIT_PARQUET is None:
    fits_path = hdf5_path.parent / f'{hdf5_path.stem}_fits.parquet'
else:
    fits_path = Path(FIT_PARQUET)
if not fits_path.exists():
    raise FileNotFoundError(f'Fit parquet not found: {fits_path}')

height_path = Path(HEIGHT_MAP_FITS)
if not height_path.exists():
    raise FileNotFoundError(f'Height map not found: {height_path}')

print(f'HDF5:        {hdf5_path}')
print(f'Fit parquet: {fits_path}')
print(f'Height map:  {height_path}')

visit_info = QTable.read(str(hdf5_path), path='visits')
aosTable = QTable.read(str(hdf5_path), path='donuts')
print(f'Loaded {len(aosTable):,} donuts, {len(aosTable.colnames)} columns, '
      f'{len(visit_info)} visits')

In [ ]:
# Load the per-visit linear-fit coefficients (k1, k2, k3 for Z4)
fits_table = pd.read_parquet(fits_path)
print(f'Fit rows: {len(fits_table)}, columns: {len(fits_table.columns)}')
print([c for c in fits_table.columns if c.startswith('z1toz3_z4_')])

required = ['day_obs', 'seq_num',
            'z1toz3_z4_c1', 'z1toz3_z4_c2', 'z1toz3_z4_c3']
missing = [c for c in required if c not in fits_table.columns]
if missing:
    raise KeyError(f'Missing required fit columns: {missing}')

if 'z1toz3_bad_fit' in fits_table.columns:
    n_bad = int(fits_table['z1toz3_bad_fit'].sum())
    print(f'  bad_fit visits: {n_bad}/{len(fits_table)}')

<a id='analysis'></a>
## Analysis

In [ ]:
# Build donut-level arrays we will re-use
zk_col = f'zk_{COORD_SYS}'
zk_intr_col = f'zk_intrinsic_{COORD_SYS}'  # pipeline's per-donut intrinsic (μm)
thx_col = f'thx_{COORD_SYS}'
thy_col = f'thy_{COORD_SYS}'

zk_arr = np.stack(aosTable[zk_col])            # (n_donuts, n_Z)
nZk = zk_arr.shape[1]
iZs = list(range(4, 4 + nZk)) if nZk != 19 else list(range(4, 23))
if 'nollIndices' in visit_info.colnames:
    iZs_vi = [int(n) for n in visit_info['nollIndices'][0]]
    if len(iZs_vi) == nZk:
        iZs = iZs_vi
iZidx = {iZ: i for i, iZ in enumerate(iZs)}
print(f'Noll indices ({nZk} terms): {iZs}')
z4_col_idx = iZidx[4]

Z4_data_um = zk_arr[:, z4_col_idx]
Z4_pipeline_intrinsic_um = np.stack(aosTable[zk_intr_col])[:, z4_col_idx]

# Field angles (linear fits use these) — stored in radians
thx_deg = np.rad2deg(np.asarray(aosTable[thx_col], dtype=float))
thy_deg = np.rad2deg(np.asarray(aosTable[thy_col], dtype=float))

# fpx / fpy in mm — derived from per-detector pixel centroids via cameraGeom.
camera = LsstCam.getCamera()
fpx, fpy = compute_fp_coords(aosTable, camera,
                             x_col='centroid_x_intra',
                             y_col='centroid_y_intra')
print(f'fpx range [mm]: [{np.nanmin(fpx):+.1f}, {np.nanmax(fpx):+.1f}]; '
      f'fpy range [mm]: [{np.nanmin(fpy):+.1f}, {np.nanmax(fpy):+.1f}]')

# Optional selection
keep = np.ones(len(aosTable), dtype=bool)
if REQUIRE_MATCHED and 'matched_intra_extra' in aosTable.colnames:
    keep &= np.asarray(aosTable['matched_intra_extra'], dtype=bool)
print(f'Selecting {keep.sum():,}/{len(keep):,} donuts')

In [ ]:
# Merge per-visit (k1, k2, k3) coefficients onto the donut table and
# compute the linear-fit Z4 prediction per donut.

fit_keys = fits_table[['day_obs', 'seq_num',
                       'z1toz3_z4_c1', 'z1toz3_z4_c2', 'z1toz3_z4_c3']].copy()

donuts_df = pd.DataFrame({
    'day_obs': np.asarray(aosTable['day_obs'], dtype=int),
    'seq_num': np.asarray(aosTable['seq_num'], dtype=int),
})
merged = donuts_df.merge(fit_keys, on=['day_obs', 'seq_num'], how='left')

k1 = merged['z1toz3_z4_c1'].to_numpy()
k2 = merged['z1toz3_z4_c2'].to_numpy()
k3 = merged['z1toz3_z4_c3'].to_numpy()
missing_fit = np.isnan(k1) | np.isnan(k2) | np.isnan(k3)
print(f'Donuts missing a linear fit: {int(missing_fit.sum())}')
keep &= ~missing_fit

Z4_linear_um = linear_fit_values(thx_deg, thy_deg, k1, k2, k3, fp_radius=FP_RADIUS_DEG)
# The linear fit was performed on (Z4_data − Z4_pipeline_intrinsic); subtracting
# only Z4_linear from Z4_data keeps the field-dependent intrinsic shape in
# the map while removing per-image piston/tilt drift.
Z4_linear_corr_um = Z4_data_um - Z4_linear_um
print('Z4 statistics (μm):')
for label, arr in [('Z4_data', Z4_data_um),
                   ('Z4_linear_model', Z4_linear_um),
                   ('Z4 - linear', Z4_linear_corr_um),
                   ('Z4_pipeline_intrinsic', Z4_pipeline_intrinsic_um)]:
    a = arr[keep]
    print(f'  {label:24s}  mean={np.nanmean(a):+.3f}  std={np.nanstd(a):.3f}  '
          f'n={len(a):,}')

In [ ]:
# Load the focal plane height map and build the KNN interpolator.
met_table = make_metrology_table(str(height_path))
print(f'Metrology points: {len(met_table):,}')
height_interp = get_height_interpolator(met_table, k=3, weight_type='distance',
                                        ztype='measured')

# Evaluate height at each donut (use intra-focal focal-plane position)
height_mm = height_interp(fpx, fpy)
print(f'Height range [mm]: [{np.nanmin(height_mm):+.4f}, {np.nanmax(height_mm):+.4f}]')

In [ ]:
# Build the ts_wep Z4 intrinsic interpolator and the custom Z4 intrinsic per donut.
band = getattr(BandLabel, INTRINSIC_BAND_NAME)
z4_um_interp, _intr_debug = build_z4_intrinsic_interpolator(
    iZs, band=band,
    extent_deg=INTRINSIC_GRID_EXTENT_DEG, n_grid=INTRINSIC_GRID_N,
)

Z4_custom_intrinsic_um, parts = build_custom_z4_intrinsic(
    thx_deg, thy_deg, fpx, fpy,
    z4_um_interp=z4_um_interp,
    height_interp=height_interp,
    height_to_z4_um_per_mm=HEIGHT_TO_Z4_UM_PER_MM,
)
Z4_optics_um = parts['optics']
Z4_height_um = parts['height']

print('Custom Z4 intrinsic components (μm):')
for label, arr in [('ts_wep optics', Z4_optics_um),
                   ('height term',   Z4_height_um),
                   ('custom total',  Z4_custom_intrinsic_um)]:
    a = arr[keep]
    print(f'  {label:14s}  mean={np.nanmean(a):+.3f}  std={np.nanstd(a):.3f}  '
          f'min={np.nanmin(a):+.3f}  max={np.nanmax(a):+.3f}')

In [ ]:
# Transposed pipeline Z4 intrinsic: for each donut, swap x↔y relative to
# its CCD center and read the pipeline intrinsic at the swapped position
# via nearest-neighbor lookup. Used to diagnose a suspected fpx↔fpy swap
# inside the pipeline's intrinsic-Zernike interpolation.
det_names_all = np.asarray(aosTable['detector']).astype(str)
Z4_pipeline_transposed_um, fpx_swap, fpy_swap, det_centers = \
    build_transposed_pipeline_z4(
        fpx, fpy, det_names_all, Z4_pipeline_intrinsic_um, camera,
        keep_mask=keep,
    )
print(f'{len(det_centers)} detector centers computed')
a = Z4_pipeline_transposed_um[keep]
print('Transposed pipeline Z4 intrinsic (μm):')
print(f'  mean={np.nanmean(a):+.3f}  std={np.nanstd(a):.3f}  '
      f'min={np.nanmin(a):+.3f}  max={np.nanmax(a):+.3f}')

<a id='results'></a>
## Results & Plots

In [ ]:
# Common binning used across the focal-plane maps
xbins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)
ybins = np.linspace(-FP_MAX_MM, FP_MAX_MM, FP_NSTEPS)

xv = fpx[keep]
yv = fpy[keep]

# Collect every figure produced below so we can save them to a single PDF.
# Re-running this cell starts a fresh collection (no stale pages).
_pdf_figs = []


def _collect(fig):
    """Remember a figure so it gets written to PDF_OUTPUT at the end."""
    _pdf_figs.append(fig)
    return fig

In [ ]:
# 1) Median Z4 (linear-corrected) in focal plane — this is what the pipeline
#    intrinsic ought to match.
stat_corr, xe, ye = bin_median_2d(xv, yv, Z4_linear_corr_um[keep], xbins, ybins)
vabs = np.nanpercentile(np.abs(stat_corr), 98)
fig, ax = plot_fp_map(
    stat_corr, xe, ye,
    title='median Z4 − (k1,k2,k3) linear fit',
    cmap='RdBu_r', vmin=-vabs, vmax=vabs,
    label='Z4 [μm]',
)
_collect(fig)

In [ ]:
# 2) Custom Z4 intrinsic (ts_wep getIntrinsicZernikes + height term) in focal plane
stat_custom, _, _ = bin_median_2d(xv, yv, Z4_custom_intrinsic_um[keep], xbins, ybins)
vabs_c = np.nanpercentile(np.abs(stat_custom), 98)
fig, ax = plot_fp_map(
    stat_custom, xe, ye,
    title='custom Z4 intrinsic: ts_wep + height map',
    cmap='RdBu_r', vmin=-vabs_c, vmax=vabs_c,
    label='Z4 [μm]',
)
_collect(fig)

In [ ]:
# 3) Difference: median (Z4 − linear) − median (custom Z4 intrinsic)
#    If the pipeline Z4 intrinsic already absorbs the CCD height variation,
#    this should look flat; any CCD-to-CCD pattern highlights what's left.
diff = stat_corr - stat_custom
vabs_d = np.nanpercentile(np.abs(diff), 98)
fig, ax = plot_fp_map(
    diff, xe, ye,
    title='median (Z4 − linear)  −  custom Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_d, vmax=vabs_d,
    label='ΔZ4 [μm]',
)
_collect(fig)

In [ ]:
# 3b) Same difference, but using the pipeline Z4 intrinsic instead of the
#     custom one: median (Z4 − linear) − pipeline Z4 intrinsic.
resid_pipe_per_donut = Z4_linear_corr_um - Z4_pipeline_intrinsic_um
stat_resid_pipe, _, _ = bin_median_2d(xv, yv, resid_pipe_per_donut[keep], xbins, ybins)
vabs_rp = np.nanpercentile(np.abs(stat_resid_pipe), 98)
fig, ax = plot_fp_map(
    stat_resid_pipe, xe, ye,
    title='median (Z4 − linear)  −  pipeline Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_rp, vmax=vabs_rp,
    label='ΔZ4 [μm]',
)
_collect(fig)

### Pipeline Z4 intrinsic vs custom Z4 intrinsic

Directly compare the per-donut Z4 intrinsic already stored in the HDF5 (computed
by the pipeline) against the custom Z4 intrinsic built from `ts_wep`'s
`Instrument.getIntrinsicZernikes` **plus** the CCD height-map term.

In [ ]:
# 4a) Pipeline Z4 intrinsic map (μm) for reference
stat_pipe, _, _ = bin_median_2d(xv, yv, Z4_pipeline_intrinsic_um[keep], xbins, ybins)
vabs_p = np.nanpercentile(np.abs(stat_pipe), 98)
fig, ax = plot_fp_map(
    stat_pipe, xe, ye,
    title='pipeline Z4 intrinsic (from HDF5 zk_intrinsic)',
    cmap='RdBu_r', vmin=-vabs_p, vmax=vabs_p,
    label='Z4 [μm]',
)
_collect(fig)

# 4b) Pipeline − custom, in focal plane
diff_pipe = stat_pipe - stat_custom
vabs_pd = np.nanpercentile(np.abs(diff_pipe), 98)
fig, ax = plot_fp_map(
    diff_pipe, xe, ye,
    title='pipeline Z4 intrinsic  −  custom Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_pd, vmax=vabs_pd,
    label='ΔZ4 [μm]',
)
_collect(fig)

In [ ]:
# 5a) Transposed pipeline Z4 intrinsic (x↔y swap per CCD), binned in (fpx, fpy)
stat_pipe_T, _, _ = bin_median_2d(xv, yv, Z4_pipeline_transposed_um[keep],
                                  xbins, ybins)
vabs_pT = np.nanpercentile(np.abs(stat_pipe_T), 98)
fig, ax = plot_fp_map(
    stat_pipe_T, xe, ye,
    title='transposed pipeline Z4 intrinsic (swap x↔y per CCD)',
    cmap='RdBu_r', vmin=-vabs_pT, vmax=vabs_pT,
    label='Z4 [μm]',
)
_collect(fig)

# 5b) Difference with the transposed pipeline intrinsic:
#     median (Z4 − linear) − transposed pipeline Z4 intrinsic.
resid_pipeT_per_donut = Z4_linear_corr_um - Z4_pipeline_transposed_um
stat_resid_pipeT, _, _ = bin_median_2d(xv, yv, resid_pipeT_per_donut[keep],
                                       xbins, ybins)
vabs_rpT = np.nanpercentile(np.abs(stat_resid_pipeT), 98)
fig, ax = plot_fp_map(
    stat_resid_pipeT, xe, ye,
    title='median (Z4 − linear)  −  transposed pipeline Z4 intrinsic',
    cmap='RdBu_r', vmin=-vabs_rpT, vmax=vabs_rpT,
    label='ΔZ4 [μm]',
)
_collect(fig)

In [ ]:
# 4c) Hexbin: pipeline vs custom (1:1 line overlaid)
pipe = Z4_pipeline_intrinsic_um[keep]
custom = Z4_custom_intrinsic_um[keep]

fig, ax = plt.subplots(figsize=(5.5, 5.5))
hb = ax.hexbin(pipe, custom, gridsize=120, mincnt=1, cmap='viridis')
lo = min(np.nanmin(pipe), np.nanmin(custom))
hi = max(np.nanmax(pipe), np.nanmax(custom))
ax.plot([lo, hi], [lo, hi], 'r-', lw=0.8, label='y = x')
ax.set_xlabel('pipeline Z4 intrinsic [μm]')
ax.set_ylabel('custom Z4 intrinsic [μm]')
ax.set_title('pipeline vs custom Z4 intrinsic (per donut)')
ax.legend(loc='upper left', frameon=True)
add_colorbar(hb).set_label('count')
fig.tight_layout()
_collect(fig)

# 4d) Hexbin: difference vs height (diagnostic for the height term)
fig, ax = plt.subplots(figsize=(5.5, 5.5))
hb = ax.hexbin(height_mm[keep], (pipe - custom), gridsize=120, mincnt=1,
               cmap='viridis')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel('local height [mm]')
ax.set_ylabel('pipeline − custom Z4 intrinsic [μm]')
ax.set_title('Does the pipeline intrinsic track the height map?')
add_colorbar(hb).set_label('count')
fig.tight_layout()
_collect(fig)

# 4e) Residuals of the linear-corrected data against each intrinsic
res_pipe = Z4_linear_corr_um[keep] - pipe
res_cust = Z4_linear_corr_um[keep] - custom
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True, sharey=True)
for ax, data, name in zip(axes, [res_pipe, res_cust],
                          ['(Z4 − linear) − pipeline intrinsic',
                           '(Z4 − linear) − custom intrinsic']):
    ax.hist(data, bins=200, histtype='step')
    ax.set_title(f'{name}\nmean={np.nanmean(data):+.3f} μm,  '
                 f'std={np.nanstd(data):.3f} μm')
    ax.set_xlabel('ΔZ4 [μm]')
axes[0].set_ylabel('donuts')
fig.tight_layout()
_collect(fig)

# Write every collected figure to a single PDF.
pdf_path = Path(PDF_OUTPUT)
pdf_path.parent.mkdir(parents=True, exist_ok=True)
with PdfPages(pdf_path) as pdf:
    for f in _pdf_figs:
        pdf.savefig(f, bbox_inches='tight')
print(f'Saved {len(_pdf_figs)} pages to {pdf_path}')